# 02 Feature Checker — CPU vs GPU 특징 비교 · 증강 시각화

feature_extractor (CPU) 와 Frontend (GPU) 결과가 일치하는지 확인하고,
증강 효과를 눈으로 검증한다.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.utils.config import load_config
from src.data.preprocessing import load_audio, normalize_wave, fix_length
from src.data.feature_extractor import extract_feature, make_channels
from src.models.frontend import Frontend

cfg = load_config('../configs/exp001_baseline.yaml')
AUDIO_DIR = Path('..') / cfg.data.audio_dir

import pandas as pd
df = pd.read_csv(Path('..') / cfg.data.train_csv)
sample_path = AUDIO_DIR / str(df.iloc[0][cfg.data.id_col])
print(f'샘플 파일: {sample_path}')

## 1. CPU 특징 (feature_extractor)

In [ ]:
target_len = int(cfg.data.sample_rate * cfg.data.duration)
wav = load_audio(sample_path, cfg.data.sample_rate)
wav = normalize_wave(wav)
wav = fix_length(wav, target_len, mode='center')

S_cpu = extract_feature(wav, cfg)          # (H, W)
arr_cpu = make_channels(S_cpu, cfg)        # (C, H, W)

print(f'CPU feature shape: {arr_cpu.shape}')
print(f'min={arr_cpu.min():.3f}, max={arr_cpu.max():.3f}, mean={arr_cpu.mean():.3f}')

plt.figure(figsize=(10, 3))
plt.imshow(arr_cpu[0], aspect='auto', origin='lower', cmap='viridis')
plt.title('CPU feature (channel 0)')
plt.colorbar()
plt.tight_layout()
plt.show()

## 2. GPU 특징 (Frontend)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
frontend = Frontend(cfg).to(device).eval()

wav_t = torch.from_numpy(wav).unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, T)

with torch.no_grad():
    arr_gpu = frontend(wav_t)[0].cpu().numpy()  # (C, H, W)

print(f'GPU feature shape: {arr_gpu.shape}')
print(f'min={arr_gpu.min():.3f}, max={arr_gpu.max():.3f}, mean={arr_gpu.mean():.3f}')

plt.figure(figsize=(10, 3))
plt.imshow(arr_gpu[0], aspect='auto', origin='lower', cmap='viridis')
plt.title('GPU feature (channel 0)')
plt.colorbar()
plt.tight_layout()
plt.show()

## 3. CPU vs GPU 차이 확인

In [ ]:
# image_size가 다를 수 있으므로 GPU 결과를 CPU shape로 리사이즈 후 비교
from PIL import Image

H, W = arr_cpu.shape[1], arr_cpu.shape[2]
arr_gpu_resized = np.stack([
    np.array(Image.fromarray(arr_gpu[c]).resize((W, H), Image.BILINEAR))
    for c in range(arr_gpu.shape[0])
])

diff = np.abs(arr_cpu - arr_gpu_resized)
print(f'평균 절대 차이: {diff.mean():.4f}')
print(f'최대 절대 차이: {diff.max():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(arr_cpu[0], aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('CPU')
axes[1].imshow(arr_gpu_resized[0], aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('GPU')
im = axes[2].imshow(diff[0], aspect='auto', origin='lower', cmap='hot')
axes[2].set_title('|CPU - GPU|')
plt.colorbar(im, ax=axes[2])
plt.tight_layout()
plt.show()

## 4. 증강 효과 시각화

In [ ]:
from src.data.augment import apply_waveform_aug
from omegaconf import OmegaConf

# 증강 ON 설정
aug_cfg = OmegaConf.merge(cfg, OmegaConf.from_dotlist([
    'augment.gain=true',
    'augment.noise=true',
    'augment.time_shift=true',
]))

n_aug = 4
fig, axes = plt.subplots(1 + n_aug, 1, figsize=(14, 3 * (1 + n_aug)))

S_orig = extract_feature(wav, cfg)
axes[0].imshow(S_orig, aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Original')

for i in range(n_aug):
    wav_aug = apply_waveform_aug(wav.copy(), aug_cfg)
    S_aug = extract_feature(wav_aug, cfg)
    axes[i + 1].imshow(S_aug, aspect='auto', origin='lower', cmap='viridis')
    axes[i + 1].set_title(f'Augmented #{i+1}')

plt.tight_layout()
plt.show()

## 5. 캐시 .npy 로드 검증

In [ ]:
cache_dir = Path('..') / 'outputs' / cfg.exp_name / 'cache'
fname = Path(str(df.iloc[0][cfg.data.id_col])).stem + '.npy'
cache_path = cache_dir / fname

if cache_path.exists():
    cached = np.load(cache_path)
    print(f'캐시 shape: {cached.shape}, dtype: {cached.dtype}')
    plt.figure(figsize=(10, 3))
    plt.imshow(cached[0], aspect='auto', origin='lower', cmap='viridis')
    plt.title('Cached feature (channel 0)')
    plt.colorbar()
    plt.tight_layout()
    plt.show()
else:
    print(f'캐시 없음: {cache_path}')
    print('먼저 scripts/cache_features.py 를 실행하세요.')